# 1. SETUP

In [1]:
import os
import sys
from pathlib import Path
import duckdb
import pandas as pd
import pandera.pandas as pa

BASE_DIR = Path(r"C:\Users\sonar\Coding\LED\promessa-ou-contradicao-de3")
os.chdir(BASE_DIR)  

sys.path.append(str(BASE_DIR / "schema"))

from silver_schema import schema_silver
from documents_schema import schema_documents

CHUNKS = "data/mock/silver_mock.parquet"
MANIFEST = "data/mock/documents_mock.csv"
CHUNKS_BROKEN = "data/mock/silver_broken.parquet"
MANIFEST_BROKEN = "data/mock/documents_broken.csv" 
PROPOSALS = "data/mock/proposals_mock.parquet"


# 2. Carregar dados

In [2]:
chunks = pd.read_parquet(CHUNKS)
manifest = pd.read_csv(MANIFEST)
chunks_b = pd.read_parquet(CHUNKS_BROKEN)
manifest_b = pd.read_csv(MANIFEST_BROKEN)

# 3. Rodar Testes


In [3]:
checagens_silver = [
    "ID do chunk duplicado",
    "O ID não existe no manifest",
    "Página menor ou igual a zero",
    "Texto vazio",
]
def testar_silver(df, schema=schema_silver, checagens=checagens_silver):
    try:
        schema.validate(df, lazy=True)
        print("Todos os checks passaram.")
    except pa.errors.SchemaErrors as err:
        checks_com_erro = set(err.failure_cases["check"].unique())
        for check in checagens:
            status = "FAIL" if check in checks_com_erro else "PASS"
            print(f"[{status}] {check}")


checagens_documents = [
    "Candidato a presidência não deve constar uma UF estadual",
    "Candidato a governador deve constar uma UF estadual válida",
    "Cargo inválido",
    "UF inválida"
]


def testar_documents(df, schema=schema_documents, checagens=checagens_documents):
    try:
        schema.validate(df, lazy=True)
        print("Todos os checks passaram.")
    except pa.errors.SchemaErrors as err:
        checks_com_erro = set(err.failure_cases["check"].unique())
        for check in checagens:
            status = "FAIL" if check in checks_com_erro else "PASS"
            print(f"[{status}] {check}")


testar_silver(chunks)
testar_documents(manifest)

Todos os checks passaram.
Todos os checks passaram.


In [4]:
checagens_silver = [
    "ID do chunk duplicado",
    "O ID não existe no manifest",
    "Página menor ou igual a zero",
    "Texto vazio",
]
def testar_silver(df, schema=schema_silver, checagens=checagens_silver):
    try:
        schema.validate(df, lazy=True)
        print("Todos os checks passaram.")
    except pa.errors.SchemaErrors as err:
        checks_com_erro = set(err.failure_cases["check"].unique())
        print("Erros do Silver")
        for check in checagens:
            status = "FAIL" if check in checks_com_erro else "PASS"
            print(f"[{status}] {check}")

checagens_documents = [
    "Candidato a presidência não deve constar uma UF estadual",
    "Candidato a governador deve constar uma UF estadual válida",
    "Cargo inválido",
    "UF inválida"
]

def testar_documents(df, schema=schema_documents, checagens=checagens_documents):
    try:
        schema.validate(df, lazy=True)
        print("Todos os checks passaram.")
    except pa.errors.SchemaErrors as err:
        checks_com_erro = set(err.failure_cases["check"].unique())
        print("\nErros no Documents")
        for check in checagens:
            status = "FAIL" if check in checks_com_erro else "PASS"
            print(f"[{status}] {check}")


testar_silver(chunks_b, schema_silver, checagens_silver)
testar_documents(manifest_b, schema_documents, checagens_documents)

Erros do Silver
[FAIL] ID do chunk duplicado
[FAIL] O ID não existe no manifest
[FAIL] Página menor ou igual a zero
[FAIL] Texto vazio

Erros no Documents
[PASS] Candidato a presidência não deve constar uma UF estadual
[PASS] Candidato a governador deve constar uma UF estadual válida
[FAIL] Cargo inválido
[PASS] UF inválida


# 4 Storage


In [5]:
con = duckdb.connect()

con.execute(f"CREATE OR REPLACE TABLE documents AS SELECT * FROM read_csv_auto('{MANIFEST}')")
con.execute(f"CREATE OR REPLACE TABLE chunks    AS SELECT * FROM read_parquet('{CHUNKS}')")
con.execute(f"CREATE OR REPLACE TABLE proposals    AS SELECT * FROM read_parquet('{PROPOSALS}')")
con.execute("SHOW TABLES").df()



,name
0,chunks
1,documents
2,proposals


Chunks por documento

In [6]:
con.execute("""
    SELECT d.document_id, d.candidate, COUNT(ch.chunk_id) AS n_chunks
    FROM documents d
    LEFT JOIN chunks ch ON ch.document_id = d.document_id
    GROUP BY d.document_id, d.candidate
    ORDER BY d.document_id
""").df()

,document_id,candidate,n_chunks
0,PRES_001,Ana Ribeiro Matos,5
1,PRES_002,Joaquim Torres Lemos,5
2,PRES_003,Marta Feijó Andrade,0


Chunks por página

In [7]:

con.execute("""
    SELECT document_id, page, COUNT(*) AS n_chunks
    FROM chunks
    GROUP BY document_id, page
    ORDER BY document_id, page
""").df()



,document_id,page,n_chunks
0,PRES_001,3,1
1,PRES_001,12,2
2,PRES_001,13,1
3,PRES_001,18,1
4,PRES_002,5,1
5,PRES_002,7,2
6,PRES_002,9,1
7,PRES_002,22,1


## Teste dos dados do duckdb

In [8]:


chunks_db = con.execute("SELECT * FROM chunks").df()
manifest_db = con.execute("SELECT * FROM documents").df()
testar_silver(chunks_db)
testar_documents(manifest_db)




Todos os checks passaram.
Todos os checks passaram.


In [9]:
con.close()